# 🛡️ Day 02a: Encryption, Hashing, VPN & Firewalls

---

## 🎯 What You'll Master Today
- Symmetric vs asymmetric encryption
- Hashing & digital signatures
- VPN types and tunneling
- Firewall types & rules

---

### 🎭 The Analogy

- **Symmetric encryption** = a shared locker with one key both people have  
- **Asymmetric encryption** = a mailbox anyone can drop mail into (public key), but only you can open (private key)  
- **Hashing** = a fingerprint — uniquely identifies data, can't reverse-engineer the person  
- **VPN** = a private tunnel through a public highway  

```
╔══════════════════════════════════════════════════════════════════════╗
║                 ENCRYPTION LANDSCAPE                                ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  SYMMETRIC (one key)          ASYMMETRIC (key pair)                 ║
║  ┌─────────────────┐          ┌─────────────────────────┐           ║
║  │ Same key to     │          │ Public key → encrypt    │           ║
║  │ encrypt & decrypt│          │ Private key → decrypt   │           ║
║  │                 │          │                         │           ║
║  │ AES, ChaCha20   │          │ RSA, ECC, Ed25519       │           ║
║  │ Fast! Bulk data │          │ Slow. Key exchange only │           ║
║  └─────────────────┘          └─────────────────────────┘           ║
║                                                                      ║
║  HASHING (one-way)            DIGITAL SIGNATURE                     ║
║  ┌─────────────────┐          ┌─────────────────────────┐           ║
║  │ Input → fixed   │          │ Hash + sign with        │           ║
║  │ size output     │          │ private key = signature │           ║
║  │ Can't reverse!  │          │ Verify with public key  │           ║
║  │ SHA-256, bcrypt │          │ Proves authenticity     │           ║
║  └─────────────────┘          └─────────────────────────┘           ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Symmetric vs Asymmetric Encryption Demo
# ============================================
import hashlib
import os

def xor_encrypt(data, key):
    """Simple XOR cipher (symmetric — same key encrypts and decrypts)."""
    return bytes(d ^ key[i % len(key)] for i, d in enumerate(data))

def demo_symmetric():
    print("  ═══ Symmetric Encryption (XOR Demo) ═══\n")
    
    plaintext = b"Hello, Network Security!"
    key = os.urandom(8)  # Both parties share this
    
    encrypted = xor_encrypt(plaintext, key)
    decrypted = xor_encrypt(encrypted, key)  # Same key!
    
    print(f"    Plaintext : {plaintext.decode()}")
    print(f"    Key       : {key.hex()}")
    print(f"    Encrypted : {encrypted.hex()}")
    print(f"    Decrypted : {decrypted.decode()}")
    print(f"    Match?    : {'✅' if plaintext == decrypted else '❌'}")

def demo_asymmetric():
    print("\n  ═══ Asymmetric Encryption (Simplified RSA Concept) ═══\n")
    # Using small primes for demonstration only!
    p, q = 61, 53
    n = p * q           # 3233 (public modulus)
    phi = (p-1)*(q-1)   # 3120
    e = 17              # Public exponent
    d = 2753            # Private exponent (e*d mod phi = 1)
    
    message = 42
    encrypted = pow(message, e, n)    # Encrypt with public key
    decrypted = pow(encrypted, d, n)  # Decrypt with private key
    
    print(f"    Public Key  : (e={e}, n={n})")
    print(f"    Private Key : (d={d}, n={n})")
    print(f"    Message     : {message}")
    print(f"    Encrypted   : {encrypted}")
    print(f"    Decrypted   : {decrypted}")
    print(f"    Match?      : {'✅' if message == decrypted else '❌'}")
    print(f"\n    Real RSA uses 2048+ bit keys (not 12-bit!)")

demo_symmetric()
demo_asymmetric()

In [ ]:
# ============================================
# 2. Hashing & Digital Signatures
# ============================================
import hmac

def demo_hashing():
    print("  ═══ Hashing Properties ═══\n")
    
    messages = [
        "Hello World",
        "Hello World!",   # One char difference
        "Hello World" * 100,  # Much longer
    ]
    
    for msg in messages:
        h = hashlib.sha256(msg.encode()).hexdigest()
        display = msg[:30] + ('...' if len(msg) > 30 else '')
        print(f"    Input: {display:>35} → SHA-256: {h[:32]}...")
    
    print(f"\n  Properties:")
    print(f"    ✅ Deterministic : same input → same hash")
    print(f"    ✅ Fixed output  : always 256 bits (SHA-256)")
    print(f"    ✅ Avalanche     : tiny input change → completely different hash")
    print(f"    ✅ One-way       : can't reverse hash → original")
    print(f"    ✅ Collision-free: nearly impossible for two inputs to have same hash")

def demo_digital_signature():
    print(f"\n  ═══ Digital Signature (HMAC Demo) ═══\n")
    
    secret_key = b"supersecretkey"   # Shared secret for HMAC
    message = b"Transfer $500 to Alice"
    
    # Sign
    signature = hmac.new(secret_key, message, hashlib.sha256).hexdigest()
    
    # Verify (correct message)
    verify_ok = hmac.new(secret_key, message, hashlib.sha256).hexdigest()
    
    # Verify (tampered message)
    tampered = b"Transfer $5000 to Alice"
    verify_bad = hmac.new(secret_key, tampered, hashlib.sha256).hexdigest()
    
    print(f"    Message   : {message.decode()}")
    print(f"    Signature : {signature[:32]}...")
    print(f"    Verify OK : {signature == verify_ok} ✅")
    print(f"    Tampered  : {signature == verify_bad} ❌ (signature mismatch!)")

demo_hashing()
demo_digital_signature()

In [ ]:
# ============================================
# 3. VPN Types & Tunneling
# ============================================

print("""
  ╔══════════════════════════════════════════════════════════════════════╗
  ║                         VPN TYPES                                   ║
  ╠═══════════════════╤════════════════════════════════════════════════╣
  ║ Type              │ Description                                    ║
  ╠═══════════════════╪════════════════════════════════════════════════╣
  ║ Site-to-Site      │ Connects two entire networks (office to       ║
  ║                   │ office). Always-on. Uses IPsec.               ║
  ╠═══════════════════╪════════════════════════════════════════════════╣
  ║ Remote Access     │ Single user → corporate network.              ║
  ║                   │ Employee working from home.                    ║
  ╠═══════════════════╪════════════════════════════════════════════════╣
  ║ SSL/TLS VPN       │ Browser-based. No special client needed.      ║
  ║                   │ Uses HTTPS (port 443).                         ║
  ╚═══════════════════╧════════════════════════════════════════════════╝

  VPN Protocols:
  ┌──────────────┬──────────┬───────────────────────────────────────┐
  │ Protocol     │ Security │ Notes                                 │
  ├──────────────┼──────────┼───────────────────────────────────────┤
  │ IPsec        │ ✅ High  │ Network layer. Site-to-site standard │
  │ OpenVPN      │ ✅ High  │ SSL/TLS based. Open source. Flexible │
  │ WireGuard    │ ✅ High  │ Modern. Fast. Simple. ~4000 LOC     │
  │ L2TP/IPsec   │ ✅ Good  │ L2TP for tunnel + IPsec for security│
  │ PPTP         │ ❌ Weak  │ Deprecated. MS-CHAPv2 is broken     │
  └──────────────┴──────────┴───────────────────────────────────────┘

  How VPN Tunneling Works:
  ┌─────────────────────────────────────────────────┐
  │  Original:  [IP Header][TCP][Data]              │
  │  Tunneled:  [New IP][VPN Header][Encrypted      │
  │             (Original IP + TCP + Data)]         │
  │  → Original packet wrapped & encrypted inside   │
  └─────────────────────────────────────────────────┘
""")

In [ ]:
# ============================================
# 4. Firewall Types & Rules
# ============================================

class Firewall:
    def __init__(self):
        self.rules = []  # (action, protocol, src_ip, dst_port, desc)
        self.default = 'DENY'  # Default deny = whitelist model
    
    def add_rule(self, action, protocol, src_ip, dst_port, desc):
        self.rules.append((action, protocol, src_ip, dst_port, desc))
    
    def check(self, protocol, src_ip, dst_port):
        for action, rule_proto, rule_src, rule_port, desc in self.rules:
            proto_match = rule_proto == '*' or rule_proto == protocol
            src_match = rule_src == '*' or rule_src == src_ip
            port_match = rule_port == '*' or rule_port == dst_port
            
            if proto_match and src_match and port_match:
                return action, desc
        
        return self.default, 'Default policy'
    
    def show_rules(self):
        print(f"  ═══ Firewall Rules (default: {self.default}) ═══\n")
        print(f"  {'#':>3} {'Action':>6} {'Proto':>6} {'Source':>12} {'Port':>6} {'Description'}")
        print(f"  {'─'*3} {'─'*6} {'─'*6} {'─'*12} {'─'*6} {'─'*20}")
        for i, (action, proto, src, port, desc) in enumerate(self.rules, 1):
            symbol = '✅' if action == 'ALLOW' else '❌'
            print(f"  {i:>3} {symbol}{action:>5} {proto:>6} {src:>12} {str(port):>6} {desc}")

fw = Firewall()
fw.add_rule('ALLOW', 'TCP', '*',          443,  'HTTPS from anywhere')
fw.add_rule('ALLOW', 'TCP', '*',          80,   'HTTP from anywhere')
fw.add_rule('ALLOW', 'TCP', '10.0.0.0',  22,   'SSH from internal only')
fw.add_rule('DENY',  'TCP', '*',          22,   'Block SSH from outside')
fw.add_rule('ALLOW', 'UDP', '*',          53,   'DNS queries')
fw.add_rule('DENY',  '*',   '*',          '*',  'Block everything else')

fw.show_rules()

# Test some packets
print(f"\n  ═══ Packet Filtering ═══\n")
tests = [
    ('TCP', '8.8.8.8',  443),
    ('TCP', '10.0.0.0', 22),
    ('TCP', '1.2.3.4',  22),
    ('TCP', '5.6.7.8',  3306),  # MySQL
]
for proto, src, port in tests:
    action, desc = fw.check(proto, src, port)
    symbol = '✅' if action == 'ALLOW' else '❌'
    print(f"    {proto} {src}:{port} → {symbol} {action} ({desc})")

In [ ]:
# ============================================
# 5. Encryption Algorithm Reference
# ============================================

print("""
  ╔══════════════════════════════════════════════════════════════════════╗
  ║                  ENCRYPTION ALGORITHMS CHEAT SHEET                  ║
  ╠═════════════════╤══════════╤═════════╤═════════════════════════════╣
  ║ Algorithm       │ Type     │ Key Size│ Use Case                    ║
  ╠═════════════════╪══════════╪═════════╪═════════════════════════════╣
  ║ AES-256-GCM     │ Symmetric│ 256 bit │ TLS data, disk encryption  ║
  ║ ChaCha20-Poly   │ Symmetric│ 256 bit │ TLS (mobile-friendly)      ║
  ║ RSA             │ Asymmetric│ 2048+  │ Key exchange, signatures   ║
  ║ ECDSA (P-256)   │ Asymmetric│ 256    │ TLS certs, faster than RSA ║
  ║ Ed25519         │ Asymmetric│ 256    │ SSH keys, modern sigs      ║
  ║ X25519          │ Key Exch │ 256     │ ECDH key exchange (TLS 1.3)║
  ║ SHA-256         │ Hash     │ 256 out │ Integrity, certs, Bitcoin  ║
  ║ SHA-3           │ Hash     │ Variable│ Next-gen hash standard     ║
  ║ bcrypt          │ Hash     │ Variable│ Password hashing (salted)  ║
  ║ Argon2          │ Hash     │ Variable│ Best password hash (memory)║
  ╚═════════════════╧══════════╧═════════╧═════════════════════════════╝

  Common Attack Types:
    • Man-in-the-Middle (MITM): intercepting communication → TLS prevents
    • Replay Attack: resending captured packets → timestamps/nonces prevent
    • DDoS: overwhelming with traffic → firewalls + rate limiting
    • SQL Injection: code in input → parameterized queries prevent
    • XSS: malicious scripts → input sanitization prevents
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Symmetric vs asymmetric? | Symmetric: one key, fast (AES). Asymmetric: key pair, slow (RSA). TLS uses both |
| 2 | Hashing vs encryption? | Hash: one-way, no key, fixed output. Encryption: reversible with key |
| 3 | What is a digital signature? | Hash the message → sign hash with private key. Verify with public key. Proves integrity + identity |
| 4 | How does a VPN work? | Creates encrypted tunnel. Original packet wrapped in new IP header + encrypted |
| 5 | Stateful vs stateless firewall? | Stateless: checks each packet independently. Stateful: tracks connection state (TCP handshake) |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Symmetric = fast (AES), Asymmetric = key exchange     │
## │  • Hashing = one-way fingerprint (SHA-256, bcrypt)       │
## │  • Digital signatures = hash + sign with private key     │
## │  • VPN = encrypted tunnel (IPsec, WireGuard, OpenVPN)    │
## │  • Firewall: default-deny, whitelist allowed traffic     │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## 🎉 CN Complete! All 16 notebooks finished!